# 🧹 Credit Risk — Feature Engineering

## 📌 Overview

This section summarizes the derived features created to improve the predictive power of the credit risk model. The transformations focus on repayment capacity, historical behavior, and customer segmentation.

- Features created: 

| Feature                 | Type        | Description                                                                   | Risk Intuition                             |
| ----------------------- | ----------- | ----------------------------------------------------------------------------- | ------------------------------------------ |
| `income_per_dependent`  | Numerical   | Monthly income divided by number of dependents (+1 to avoid division by zero) | Lower value → higher financial burden      |
| `utilization_capped`    | Numerical   | Credit utilization clipped between 0 and 1                                    | Reduces impact of extreme outliers         |
| `CreditHistoryLength`   | Numerical   | Age - 18 (proxy for credit history length)                                    | Longer history → lower risk                |
| `TotalPastDue`          | Numerical   | Total number of past due occurrences across all categories                    | More delays → higher risk                  |
| `weighted_late_score`   | Numerical   | Weighted score of late payments by severity                                   | Heavier penalties for severe delinquencies |
| `HasSeriousDelinquency` | Binary      | 1 if any 90+ days past due                                                    | Strong default indicator                   |
| `high_utilization_flag` | Binary      | 1 if utilization > 80%                                                        | High utilization → elevated risk           |
| `AgeGroup`              | Categorical | Age binned into groups                                                        | Captures lifecycle effects                 |
| `IncomeGroup`           | Categorical | Income quartiles                                                              | Socioeconomic segmentation                 |
| `DTICategory`           | Categorical | Debt-to-Income ratio categories                                               | Higher DTI → lower repayment capacity      |





## 📚 Libraries


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

## 📂 Load Dataset

In [2]:
path = Path("Data")
data = pd.read_csv(path / "cs-training-cleaned.csv") 
df_clean = data.copy()

## ⚙️ Variables Creation

In [3]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies feature engineering transformations to the input DataFrame.

    Parameters:
        df (pd.DataFrame): Cleaned input dataset

    Returns:
        pd.DataFrame: Dataset with new engineered features
    """
    
    df = df.copy()

    # --- Basic Ratios ---
    df["income_per_dependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
    df["utilization_capped"] = df["RevolvingUtilizationOfUnsecuredLines"].clip(0, 1)

    # --- Credit History ---
    df["CreditHistoryLength"] = df["age"] - 18  # assumption: credit starts at 21

    # --- Delinquency Features ---
    df["TotalPastDue"] = (
        df["NumberOfTime30-59DaysPastDueNotWorse"] +
        df["NumberOfTime60-89DaysPastDueNotWorse"] +
        df["NumberOfTimes90DaysLate"]
    )

    df["weighted_late_score"] = (
        1.0 * df["NumberOfTime30-59DaysPastDueNotWorse"] +
        1.5 * df["NumberOfTime60-89DaysPastDueNotWorse"] +
        2.5 * df["NumberOfTimes90DaysLate"]
    )

    df["HasSeriousDelinquency"] = (df["NumberOfTimes90DaysLate"] > 0).astype(int)

    # --- Utilization Flag ---
    df["high_utilization_flag"] = (df["utilization_capped"] > 0.8).astype(int)

    # --- Age Groups ---
    df["AgeGroup"] = pd.cut(
        df["age"],
        bins=[0, 25, 35, 45, 55, 65, 100],
        labels=["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
    )

    # --- Income Groups ---
    income_filled = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
    df["IncomeGroup"] = pd.qcut(
        income_filled,
        q=4,
        labels=["Low", "Medium-Low", "Medium-High", "High"],
        duplicates="drop"
    )

    # --- Debt-to-Income Category ---
    df["DTICategory"] = pd.cut(
        df["DebtRatio"],
        bins=[0, 0.36, 0.43, 1, float("inf")],
        labels=["Low", "Moderate", "High", "Very High"],
        include_lowest=True
    )

    return df

In [4]:
df_fe = feature_engineering(df_clean)

## 🔎 Brief Summary of the complete Dataset

In [5]:
print("📊 Statistical Summary:")
df_fe.describe()

📊 Statistical Summary:


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,MonthlyIncome_missing,MonthlyIncome_log,income_per_dependent,utilization_capped,CreditHistoryLength,TotalPastDue,weighted_late_score,HasSeriousDelinquency,high_utilization_flag
count,149717.000000,149717.000000,149717.000000,149717.000000,149717.000000,1.497170e+05,149717.000000,149717.000000,149717.000000,149717.000000,149717.000000,149717.000000,149717.000000,1.497170e+05,149717.000000,149717.000000,149717.000000,149717.000000,149717.000000,149717.000000
mean,0.065978,6.058057,52.323524,0.245797,353.626931,6.491994e+03,8.468096,0.090464,1.020125,0.064829,0.738199,0.208594,8.543630,4.646182e+03,0.317985,34.323524,0.401090,0.569201,0.053895,0.166060
std,0.248244,249.991210,14.747223,0.697792,2039.678899,1.288413e+04,5.138154,0.485549,1.129976,0.330088,1.107402,0.406305,0.802139,7.970173e+03,0.348600,14.747223,1.102184,1.772189,0.225811,0.372136
min,0.000000,0.000000,21.000000,0.000000,0.000000,1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.693147,1.666667e-01,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.029776,41.000000,0.000000,0.176008,4.000000e+03,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.294300,2.201000e+03,0.029776,23.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.153496,52.000000,0.000000,0.367118,5.500000e+03,8.000000,0.000000,1.000000,0.000000,0.000000,0.000000,8.612685,4.028000e+03,0.153496,34.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.555614,63.000000,0.000000,0.869710,7.400000e+03,11.000000,0.000000,2.000000,0.000000,1.000000,0.000000,8.909370,5.500000e+03,0.555614,45.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,50708.000000,99.000000,13.000000,329664.000000,3.008750e+06,58.000000,17.000000,54.000000,11.000000,20.000000,1.000000,14.917036,1.794060e+06,1.000000,81.000000,19.000000,42.500000,1.000000,1.000000


In [6]:
df_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149717 entries, 0 to 149716
Data columns (total 23 columns):
 #   Column                                Non-Null Count   Dtype   
---  ------                                --------------   -----   
 0   SeriousDlqin2yrs                      149717 non-null  int64   
 1   RevolvingUtilizationOfUnsecuredLines  149717 non-null  float64 
 2   age                                   149717 non-null  int64   
 3   NumberOfTime30-59DaysPastDueNotWorse  149717 non-null  int64   
 4   DebtRatio                             149717 non-null  float64 
 5   MonthlyIncome                         149717 non-null  float64 
 6   NumberOfOpenCreditLinesAndLoans       149717 non-null  int64   
 7   NumberOfTimes90DaysLate               149717 non-null  int64   
 8   NumberRealEstateLoansOrLines          149717 non-null  int64   
 9   NumberOfTime60-89DaysPastDueNotWorse  149717 non-null  int64   
 10  NumberOfDependents                    149717 non-null  f

In [ ]:
# # Save complete dataset
# df_fe.to_csv(path / "german_credit_cleaned.csv", index=False)